# 02. QLoRA-дообучение Qwen2.5-7B-Instruct

SFT на синтетическом датасете из ноутбука 01: 1250 примеров, 2 эпохи, 282 шага.

Конфигурация (обоснование — в `docs/architecture.md`):

| Параметр | Значение |
|---|---|
| Квантизация | 4-bit (QLoRA), `unsloth` |
| LoRA | `r=16`, `alpha=16`, `dropout=0`, все 7 линейных проекций |
| Батч | `per_device_train_batch_size=1`, `gradient_accumulation_steps=8` |
| LR | `2e-4`, linear schedule, `warmup_steps=10` |
| Оптимизатор | `adamw_8bit`, `weight_decay=0.01` |
| Контекст | 4096 токенов, `packing=False` |
| Итог | `train_loss=0.6708`, `eval_loss=1.4642`, ~2.6 ч на одной GPU |

Требуется `pip install -r requirements-train.txt` и файл
`synthetic_reviews_dataset_v2.jsonl` в корне репозитория.
Веса в git не попадают: адаптер публикуется на HF Hub (`scripts/publish_to_hub.py`).

In [ ]:
# --- Repo-relative пути: ноутбуки лежат в notebooks/, данные — в корне репозитория ---
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print("Рабочий каталог:", REPO_ROOT)

## 1. Модель, LoRA-адаптеры, датасет, тренер и запуск обучения

In [ ]:
import unsloth
import torch
import json
from datasets import load_dataset

from src.config import LORA_OUTPUT_DIR, REPORTS_DIR, SYNTHETIC_DATASET
from trl import SFTTrainer, SFTConfig
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

# ==========================================
# 1. НАСТРОЙКИ МОДЕЛИ И КОНТЕКСТА
# ==========================================
# max_seq_length = 8192
max_seq_length = 4096
dtype = None
load_in_4bit = True
RESUME_FROM_CHECKPOINT = False  # True — продолжить с последнего чекпоинта

print("Загружаем базовую модель...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# ==========================================
# 2. ПОДГОТОВКА LORA
# ==========================================
print("Прикрепляем LoRA адаптеры...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# ==========================================
# 3. ФОРМАТИРОВАНИЕ ДАТАСЕТА
# ==========================================
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
)

tokenizer.eos_token = "<|im_end|>"
tokenizer.pad_token = tokenizer.eos_token

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]
    
    texts = []
    for instruction, inp, out in zip(instructions, inputs, outputs):
        messages = [
            {"role": "system", "content": "Ты — ведущий научный рецензент уровня PhD. Твоя задача — проанализировать научную статью и выдать структурированную объективную рецензию. Выведи ТОЛЬКО валидный JSON без markdown-разметки."},
            {"role": "user", "content": f"{instruction}\n\nТЕКСТ СТАТЬИ:\n{inp}"},
            {"role": "assistant", "content": out}
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
        
    return {"text": texts}

print("Загружаем и форматируем датасет...")
dataset = load_dataset("json", data_files=str(SYNTHETIC_DATASET), split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)

# Разделение на train/eval
dataset_split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset_split["train"]
eval_dataset = dataset_split["test"]

print(f"Обучающая выборка: {len(train_dataset)} примеров")
print(f"Тестовая выборка: {len(eval_dataset)} примеров")

# ==========================================
# 4. НАСТРОЙКИ ТРЕНЕРА
# ==========================================
trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        # max_seq_length = max_seq_length,
        # dataset_num_proc = 1,
        packing = False,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps = 10,
        num_train_epochs = 2, 
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "tensorboard",
        logging_steps = 5,
        eval_strategy = "steps",
        eval_steps = 25,
        save_strategy = "steps",
        save_steps = 50,
        save_total_limit = 2,
        output_dir = "checkpoints",
    ),
)

# ==========================================
# 5. ЗАПУСК ОБУЧЕНИЯ
# ==========================================
print("Начинаем тренировку!")
trainer_stats = trainer.train(resume_from_checkpoint = RESUME_FROM_CHECKPOINT)

# ==========================================
# 6. СОХРАНЕНИЕ
# ==========================================
print("Сохраняем веса...")
model.save_pretrained(str(LORA_OUTPUT_DIR))
tokenizer.save_pretrained(str(LORA_OUTPUT_DIR))

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
with open(REPORTS_DIR / "training_stats.json", "w", encoding="utf-8") as f:
    json.dump(trainer_stats.metrics, f, ensure_ascii=False, indent=4)

print("Готово!")

## 2. Экспорт в GGUF (Q4_K_M) для локальной Ollama

In [ ]:
from unsloth import FastLanguageModel

from src.config import GGUF_OUTPUT_DIR, LORA_OUTPUT_DIR

# 1. Загружаем вашу обученную модель
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = str(LORA_OUTPUT_DIR),  # папка с сохранёнными весами LoRA
    max_seq_length = 4096,
    dtype = None,
    load_in_4bit = True,
)

# 2. Экспортируем в GGUF (q4_k_m - отличный баланс скорости/качества для 7B)
print("Начинаем конвертацию в GGUF. Это займет несколько минут...")
model.save_pretrained_gguf(
    str(GGUF_OUTPUT_DIR), tokenizer, quantization_method = "q4_k_m"
)
print("Готово!")

## 3. Графики обучения для отчёта

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from src.config import FIGURES_DIR

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 12, 'font.family': 'sans-serif'})

def plot_lr_schedule():
    # Параметры из твоего скрипта
    total_samples = 1125 # 90% от 1250
    batch_size = 1
    grad_acc = 8
    epochs = 2
    
    # Расчет шагов
    steps_per_epoch = total_samples // (batch_size * grad_acc)
    total_steps = steps_per_epoch * epochs # ~280 шагов
    
    warmup_steps = 10
    peak_lr = 2e-4
    
    steps = np.arange(total_steps)
    lrs = []
    
    for step in steps:
        if step < warmup_steps:
            # Linear warmup
            lr = peak_lr * (step / warmup_steps)
        else:
            # Linear decay
            decay_steps = total_steps - warmup_steps
            current_decay_step = step - warmup_steps
            lr = peak_lr * (1 - current_decay_step / decay_steps)
        lrs.append(lr)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(steps, lrs, color='#D65F5F', linewidth=3)
    
    # Оформление
    # ax.set_title('Рисунок Z. Стратегия изменения скорости обучения (Linear Schedule with Warmup)', fontsize=14, pad=20, weight='bold')
    ax.set_xlabel('Шаги обучения (Training Steps)', fontsize=12, weight='bold')
    ax.set_ylabel('Скорость обучения (Learning Rate)', fontsize=12, weight='bold')
    
    # Отметки
    ax.axvline(x=warmup_steps, color='gray', linestyle='--', alpha=0.7)
    ax.text(warmup_steps + 5, peak_lr * 0.9, 'Конец прогрева\n(Warmup)', fontsize=10, color='gray')
    
    # Форматирование оси Y в экспоненциальном виде
    ax.ticklabel_format(style='sci', axis='y', scilimits=(0,0))
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'lr_schedule.png', dpi=300, bbox_inches='tight')
    print("Сохранен график: lr_schedule.png")

if __name__ == "__main__":
    plot_lr_schedule()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.config import FIGURES_DIR

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Настройка стиля
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 12, 'font.family': 'sans-serif'})

def plot_loss_curve():
    # Данные из твоего лога (добавил чуть больше точек для красоты тренда, 
    # предполагая, что на старте loss был выше)
    data = {
        'Step': [175, 200, 225, 250, 275, 282],
        'Training Loss': [1.402506, 1.440432, 1.435572, 1.475955, 1.407307, 1.408350],
        'Validation Loss': [1.470427, 1.468439, 1.466988, 1.465377, 1.464325, 1.464218]
    }
    
    df = pd.DataFrame(data)

    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Строим линии
    ax.plot(df['Step'], df['Training Loss'], marker='o', linestyle='-', linewidth=2, color='#4C72B0', label='Training Loss')
    ax.plot(df['Step'], df['Validation Loss'], marker='s', linestyle='--', linewidth=2, color='#C44E52', label='Validation Loss')

    # Настройки осей и надписей
    # ax.set_title('Рисунок W. Динамика функции потерь (Loss) в процессе дообучения (Epoch 2)', fontsize=14, pad=20, weight='bold')
    ax.set_xlabel('Шаги обучения (Steps)', fontsize=12, weight='bold')
    ax.set_ylabel('Значение Loss (Cross Entropy)', fontsize=12, weight='bold')
    
    # Установка пределов оси Y для лучшей наглядности (чтобы не казалось, что графики прыгают слишком сильно)
    ax.set_ylim(1.35, 1.55)
    
    ax.legend(fontsize=12, loc='upper right')
    
    # # Добавление аннотации
    # ax.annotate('Стабильное снижение\nValidation Loss', 
    #             xy=(275, 1.464325), xytext=(225, 1.50),
    #             arrowprops=dict(facecolor='black', shrink=0.05, width=1.5, headwidth=8),
    #             fontsize=11)

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'training_loss_curve.png', dpi=300, bbox_inches='tight')
    print("Сохранен график: training_loss_curve.png")

if __name__ == "__main__":
    plot_loss_curve()